<a href="https://colab.research.google.com/github/AlKhrisW/JTIntern/blob/qila/pemodelan/preprocessing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install & Import Library

In [1]:
!pip install openpyxl -q

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

✅ Library berhasil diimport


---
## 2. Upload & Load File Excel

> Upload file `PBL_DATA_filled.xlsx` saat cell di bawah dijalankan.

In [3]:
from google.colab import files

uploaded = files.upload()  # Upload file PBL_DATA_filled.xlsx
file_name = list(uploaded.keys())[0]
print(f'📁 File diupload: {file_name}')

Saving PBL_DATA_filled.xlsx to PBL_DATA_filled.xlsx
📁 File diupload: PBL_DATA_filled.xlsx


In [4]:
# Load semua sheet sekaligus
all_sheets = pd.read_excel(file_name, sheet_name=None)

df_mahasiswa  = all_sheets['MAHASISWA'].copy()
df_perusahaan = all_sheets['PERUSAHAAN'].copy()

print(f'Sheet MAHASISWA  : {df_mahasiswa.shape[0]} baris, {df_mahasiswa.shape[1]} kolom')
print(f'Sheet PERUSAHAAN : {df_perusahaan.shape[0]} baris, {df_perusahaan.shape[1]} kolom')

Sheet MAHASISWA  : 446 baris, 7 kolom
Sheet PERUSAHAAN : 30 baris, 10 kolom


---
## 3. Eksplorasi Awal Data (Sebelum Preprocessing)

In [5]:
print('=== MAHASISWA — 5 baris pertama ===')
df_mahasiswa.head()

=== MAHASISWA — 5 baris pertama ===


,nim,nama,prodi,Teknologi yang digunakan,Minat Bidang,Skill,IPK
0,848822.862355,Dinda SIta,D4 Teknik Informatika,"Frontend: HTML, CSS, JavaScript, TailwindCSS, ...","Software Developer, Artificial Intelligence, D...","Python, JavaScript, TypeScript, Java, SQL, Tai...",3.61
1,382290.839016,Rara Gunawan,D4 Teknik Informatika,NaN,NaN,NaN,3.30
2,455004.821813,Wahyu Lestari,D4 Teknik Informatika,NaN,NaN,NaN,3.66
3,242443.569480,Hadi Putri,D4 Teknik Informatika,Laravel;NestJs;Wordpress;IoT,Software Developer,"JavaScript, Laravel, NestJS",3.87
4,305358.666040,Yanti Supriyanto,D4 Teknik Informatika,NaN,NaN,NaN,3.18


In [6]:
print('=== MAHASISWA — Info Kolom ===')
df_mahasiswa.info()

=== MAHASISWA — Info Kolom ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 446 entries, 0 to 445
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   nim                       446 non-null    float64
 1   nama                      446 non-null    object 
 2   prodi                     446 non-null    object 
 3   Teknologi yang digunakan  306 non-null    object 
 4   Minat Bidang              306 non-null    object 
 5   Skill                     306 non-null    object 
 6   IPK                       446 non-null    float64
dtypes: float64(2), object(5)
memory usage: 24.5+ KB


In [7]:
print('=== MAHASISWA — Jumlah Null per Kolom ===')
df_mahasiswa.isnull().sum()

=== MAHASISWA — Jumlah Null per Kolom ===


,0
nim,0
nama,0
prodi,0
Teknologi yang digunakan,140
Minat Bidang,140
Skill,140
IPK,0


In [8]:
print('=== PERUSAHAAN — 5 baris pertama ===')
df_perusahaan.head()

=== PERUSAHAAN — 5 baris pertama ===


,Nama Perusahaan,Profil Perusahaan,Bidang Industri (BUMN/Swasta),Posisi Magang,Minimal IPK,Job Description,Skill yang Dibutuhkan,Tools/Teknologi yang Digunakan,Durasi Magang (bulan),Status Magang (Paid/Unpaid)
0,PT Indoprima Gemilang,PT Indoprima Gemilang berdiri sejak 1976 di Su...,Swasta Nasional – Manufaktur Otomotif,IT/System Development Intern,3.00,Membantu pengembangan dan pemeliharaan sistem ...,"Pemrograman dasar, logika sistem, komunikasi, ...","Visual Basic / VB.NET, MySQL, atau teknologi w...",6,Paid
1,Pengembangan SIPP – PT Link Apisindo Media & D...,PT Link Apisindo Media adalah perusahaan IT di...,Swasta Nasional – IT & Software Development,Backend Engineer / Frontend Engineer / AI Engi...,3.00,"Backend: pengembangan API, integrasi data, pen...","Backend: REST API, database management, system...","Backend: Laravel/FastAPI, MySQL/PostgreSQL, Do...",5,Paid
2,Pengembangan Platform Satu Peta Jatim – PT Lin...,PT Link Apisindo Media berkolaborasi dengan Di...,Swasta Nasional – IT & Geospatial System,Data Engineer / Backend Developer / Frontend D...,3.00,"Data: ETL data spasial/non-spasial, integrasi ...","Data: ETL, GIS, database. Backend: Python, RES...","PostGIS, Python (FastAPI), JavaScript, OpenLay...",5,Paid
3,PT ARM Solusi,"PT ARM Solusi berdiri sejak 2001, merupakan pe...",Swasta Nasional – IT Consulting & Software Dev...,Blockchain Engineer / Backend Developer / Data...,3.00,"Blockchain: pengembangan Hyperledger Fabric, s...","Blockchain: distributed systems, smart contrac...","Hyperledger Fabric, Python, Node.js / React, S...",5,Paid
4,Pengembangan Platform Satu Peta Jatim – Batch ...,Batch kedua dari kolaborasi PT Link Apisindo M...,Swasta Nasional – IT & Geospatial System,Data Engineer / Backend Developer / Frontend D...,3.00,"Data: ETL data spasial & non-spasial, harmonis...","ETL, GIS, Python, REST API, JavaScript, WebGIS...","PostGIS, Python (FastAPI), OpenLayers/Leaflet,...",6,Paid


In [9]:
print('=== PERUSAHAAN — Info Kolom ===')
df_perusahaan.info()

=== PERUSAHAAN — Info Kolom ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Nama Perusahaan                 30 non-null     object
 1   Profil Perusahaan               30 non-null     object
 2   Bidang Industri (BUMN/Swasta)   30 non-null     object
 3   Posisi Magang                   30 non-null     object
 4   Minimal IPK                     30 non-null     object
 5   Job Description                 30 non-null     object
 6   Skill yang Dibutuhkan           30 non-null     object
 7   Tools/Teknologi yang Digunakan  30 non-null     object
 8   Durasi Magang (bulan)           30 non-null     int64 
 9   Status Magang (Paid/Unpaid)     30 non-null     object
dtypes: int64(1), object(9)
memory usage: 2.5+ KB


In [10]:
print('=== PERUSAHAAN — Jumlah Null per Kolom ===')
df_perusahaan.isnull().sum()

=== PERUSAHAAN — Jumlah Null per Kolom ===


,0
Nama Perusahaan,0
Profil Perusahaan,0
Bidang Industri (BUMN/Swasta),0
Posisi Magang,0
Minimal IPK,0
Job Description,0
Skill yang Dibutuhkan,0
Tools/Teknologi yang Digunakan,0
Durasi Magang (bulan),0
Status Magang (Paid/Unpaid),0


---
## 4. Preprocessing Sheet MAHASISWA

### 4.1 Drop baris yang memiliki nilai null pada Skill, Teknologi, atau Minat Bidang

In [11]:
before = len(df_mahasiswa)

df_mahasiswa.dropna(
    subset=['Skill', 'Teknologi yang digunakan', 'Minat Bidang'],
    inplace=True
)

df_mahasiswa.reset_index(drop=True, inplace=True)
after = len(df_mahasiswa)

print(f'Sebelum : {before} baris')
print(f'Dihapus : {before - after} baris (Skill / Teknologi / Minat Bidang = null)')
print(f'Setelah : {after} baris')

Sebelum : 446 baris
Dihapus : 141 baris (Skill / Teknologi / Minat Bidang = null)
Setelah : 305 baris


### 4.2 Perbaiki format NIM (float → string integer bersih)

In [12]:
# NIM tersimpan sebagai float (misal: 848822.862355), ambil bagian depan sebelum titik desimal
print('Contoh NIM sebelum :', df_mahasiswa['nim'].head(3).tolist())

df_mahasiswa['nim'] = df_mahasiswa['nim'].apply(
    lambda x: str(int(round(float(str(x).split('.')[0])))) if pd.notna(x) else x
)

print('Contoh NIM sesudah  :', df_mahasiswa['nim'].head(3).tolist())

Contoh NIM sebelum : [848822.8623547651, 242443.5694796109, 665049.4491126052]
Contoh NIM sesudah  : ['848822', '242443', '665049']


### 4.3 Rename kolom agar konsisten

In [13]:
df_mahasiswa.rename(columns={
    'nim'                      : 'NIM',
    'nama'                     : 'Nama',
    'prodi'                    : 'Prodi',
    'Teknologi yang digunakan' : 'Teknologi',
    'Minat Bidang'             : 'Minat_Bidang',
    'Skill'                    : 'Skill',
    'IPK'                      : 'IPK'
}, inplace=True)

print('Kolom setelah rename:', list(df_mahasiswa.columns))

Kolom setelah rename: ['NIM', 'Nama', 'Prodi', 'Teknologi', 'Minat_Bidang', 'Skill', 'IPK']


### 4.4 Trim whitespace pada kolom bertipe string

In [14]:
str_cols = ['Nama', 'Prodi', 'Teknologi', 'Minat_Bidang', 'Skill']
for col in str_cols:
    df_mahasiswa[col] = df_mahasiswa[col].str.strip()

print('✅ Whitespace dibersihkan pada:', str_cols)

✅ Whitespace dibersihkan pada: ['Nama', 'Prodi', 'Teknologi', 'Minat_Bidang', 'Skill']


### 4.5 Normalisasi separator pada Skill, Teknologi, dan Minat Bidang

> Beberapa baris menggunakan `;` atau newline sebagai pemisah. Kita standarisasi semua menjadi `, ` (koma spasi).

In [15]:
def normalize_separator(val):
    """Ubah separator ; dan newline menjadi koma, lalu bersihkan spasi berlebih."""
    if pd.isna(val):
        return val
    val = re.sub(r'[;\n]', ',', val)           # ganti ; dan newline → koma
    items = [x.strip() for x in val.split(',') if x.strip()]  # split & trim
    return ', '.join(items)

df_mahasiswa['Skill']        = df_mahasiswa['Skill'].apply(normalize_separator)
df_mahasiswa['Teknologi']    = df_mahasiswa['Teknologi'].apply(normalize_separator)
df_mahasiswa['Minat_Bidang'] = df_mahasiswa['Minat_Bidang'].apply(normalize_separator)

print('✅ Separator dinormalisasi')
print('\nContoh Skill    :', df_mahasiswa['Skill'].iloc[0])
print('Contoh Teknologi:', df_mahasiswa['Teknologi'].iloc[0])
print('Contoh Minat    :', df_mahasiswa['Minat_Bidang'].iloc[0])

✅ Separator dinormalisasi

Contoh Skill    : Python, JavaScript, TypeScript, Java, SQL, Tailwind CSS, Bootstrap, HTML & CSS, MySQL, Git & GitHub, Figma, Canva, UI/UX Design
Contoh Teknologi: Frontend: HTML, CSS, JavaScript, TailwindCSS, Bootstrap, UI/UX Design: Figma, Prototyping, Wireframing, User Flow, Database: MySQL, Programming Languages: Python, SQL, Tools & Workflow: Git, GitHub, Visual Studio Code, Canva
Contoh Minat    : Software Developer, Artificial Intelligence, Data Technology, Multimedia and game, Information System, Bussines analyst


### 4.6 Perbaiki Typo 'Bussines analyst' di Minat Bidang

In [16]:
before_typo_count = df_mahasiswa['Minat_Bidang'].str.contains('Bussines analyst', na=False).sum()
df_mahasiswa['Minat_Bidang'] = df_mahasiswa['Minat_Bidang'].str.replace('Bussines analyst', 'Business analyst', regex=False)
after_typo_count = df_mahasiswa['Minat_Bidang'].str.contains('Bussines analyst', na=False).sum()

print(f'Sebelum perbaikan: {before_typo_count} typo ditemukan.')
print(f'Setelah perbaikan: {after_typo_count} typo tersisa.')
print('✅ Typo \'Bussines analyst\' telah diperbaiki menjadi \'Business analyst\'.')

Sebelum perbaikan: 243 typo ditemukan.
Setelah perbaikan: 0 typo tersisa.
✅ Typo 'Bussines analyst' telah diperbaiki menjadi 'Business analyst'.


### 4.7 Bersihkan kategori di kolom 'Teknologi'

In [17]:
def remove_category_prefixes(text):
    if pd.isna(text):
        return text
    # Split the string by the main separator (', ') to get individual items
    items = [item.strip() for item in text.split(',')]
    cleaned_items = []
    for item in items:
        # Use a more robust regex to remove any text followed by a colon and space at the beginning of an item
        cleaned_item = re.sub(r'^[^:]+:\s*', '', item).strip()
        if cleaned_item: # Ensure we don't add empty strings if a prefix was the only content
            cleaned_items.append(cleaned_item)
    return ', '.join(cleaned_items)

df_mahasiswa['Teknologi'] = df_mahasiswa['Teknologi'].apply(remove_category_prefixes)

print('✅ Kategori di kolom Teknologi telah dibersihkan.')
print('\nContoh Teknologi setelah dibersihkan:', df_mahasiswa['Teknologi'].iloc[0])

✅ Kategori di kolom Teknologi telah dibersihkan.

Contoh Teknologi setelah dibersihkan: HTML, CSS, JavaScript, TailwindCSS, Bootstrap, Figma, Prototyping, Wireframing, User Flow, MySQL, Python, SQL, Git, GitHub, Visual Studio Code, Canva


### 4.8 Validasi & pembulatan IPK

In [18]:
# Cek IPK di luar range 0–4
invalid = df_mahasiswa[(df_mahasiswa['IPK'] < 0) | (df_mahasiswa['IPK'] > 4)]
print(f'Baris IPK tidak valid (< 0 atau > 4): {len(invalid)}')

# Bulatkan 2 desimal
df_mahasiswa['IPK'] = df_mahasiswa['IPK'].round(2)

print(f'IPK range: {df_mahasiswa["IPK"].min()} – {df_mahasiswa["IPK"].max()}')

Baris IPK tidak valid (< 0 atau > 4): 0
IPK range: 2.43 – 3.97


### 4.9 Cek & hapus duplikat NIM

In [19]:
dup = df_mahasiswa.duplicated(subset=['NIM'], keep=False)
print(f'Baris dengan NIM duplikat: {dup.sum()}')

if dup.sum() > 0:
    print('Contoh duplikat:')
    display(df_mahasiswa[dup][['NIM','Nama']].head())
    df_mahasiswa.drop_duplicates(subset=['NIM'], keep='first', inplace=True)
    df_mahasiswa.reset_index(drop=True, inplace=True)
    print(f'✅ Duplikat dihapus. Baris sekarang: {len(df_mahasiswa)}')
else:
    print('✅ Tidak ada NIM duplikat')

Baris dengan NIM duplikat: 0
✅ Tidak ada NIM duplikat


### 4.10 Preview hasil akhir MAHASISWA

In [20]:
print(f'Total baris akhir  : {df_mahasiswa.shape[0]}')
print(f'Total null tersisa : {df_mahasiswa.isnull().sum().sum()}')
print()
print('Distribusi Prodi:')
print(df_mahasiswa['Prodi'].value_counts())
df_mahasiswa.head()

Total baris akhir  : 305
Total null tersisa : 0

Distribusi Prodi:
Prodi
D4 Teknik Informatika         192
D4 Sistem Informasi Bisnis    113
Name: count, dtype: int64


,NIM,Nama,Prodi,Teknologi,Minat_Bidang,Skill,IPK
0,848822,Dinda SIta,D4 Teknik Informatika,"HTML, CSS, JavaScript, TailwindCSS, Bootstrap,...","Software Developer, Artificial Intelligence, D...","Python, JavaScript, TypeScript, Java, SQL, Tai...",3.61
1,242443,Hadi Putri,D4 Teknik Informatika,"Laravel, NestJs, Wordpress, IoT",Software Developer,"JavaScript, Laravel, NestJS",3.87
2,665049,Rizky Anggraini,D4 Teknik Informatika,"Python, JavaScript, TypeScript, PHP, SQL, Djan...","Software Developer, Artificial Intelligence, D...","Python, JavaScript, TypeScript, PHP, Java, SQL...",3.90
3,149750,Zaki Kristanti,D4 Teknik Informatika,"DiscordJS, JavaScript, PHP, Laravel, HTML, CSS...","Software Developer, Artificial Intelligence, M...","JavaScript, PHP, Java, Lua, HTML & CSS, Larave...",3.75
4,842443,Era Basuki,D4 Teknik Informatika,"PHP, Python, CSS, Java, MySQL, Cisco, Subnetti...","Software Developer, Network and Security, Data...","Python, PHP, Java, SQL, Bash / Shell, HTML & C...",3.59


---
## 5. Preprocessing Sheet PERUSAHAAN

### 5.1 Rename kolom

In [21]:
df_perusahaan.rename(columns={
    'Nama Perusahaan'                : 'Nama_Perusahaan',
    'Profil Perusahaan'              : 'Profil_Perusahaan',
    'Bidang Industri (BUMN/Swasta)'  : 'Bidang_Industri',
    'Posisi Magang'                  : 'Posisi_Magang',
    'Minimal IPK'                    : 'Minimal_IPK',
    'Job Description'                : 'Job_Description',
    'Skill yang Dibutuhkan'          : 'Skill_Dibutuhkan',
    'Tools/Teknologi yang Digunakan' : 'Teknologi_Digunakan',
    'Durasi Magang (bulan)'          : 'Durasi_Bulan',
    'Status Magang (Paid/Unpaid)'    : 'Status_Paid'
}, inplace=True)

print('Kolom setelah rename:', list(df_perusahaan.columns))

Kolom setelah rename: ['Nama_Perusahaan', 'Profil_Perusahaan', 'Bidang_Industri', 'Posisi_Magang', 'Minimal_IPK', 'Job_Description', 'Skill_Dibutuhkan', 'Teknologi_Digunakan', 'Durasi_Bulan', 'Status_Paid']


### 5.2 Tambahkan ID Perusahaan

In [22]:
df_perusahaan.insert(0, 'ID_Perusahaan', [f'P{str(i+1).zfill(3)}' for i in range(len(df_perusahaan))])
print('✅ ID_Perusahaan ditambahkan (P001 –', f'P{len(df_perusahaan):03d})')

✅ ID_Perusahaan ditambahkan (P001 – P030)


### 5.3 Bersihkan kolom Minimal_IPK

> Nilai seperti `'Tidak ada syarat khusus'` diubah menjadi `0.0` (tidak ada syarat IPK minimum).

In [23]:
print('Nilai unik Minimal_IPK sebelum:', df_perusahaan['Minimal_IPK'].unique())

# Konversi ke numerik; nilai yang tidak bisa dikonversi → NaN → isi 0
df_perusahaan['Minimal_IPK'] = pd.to_numeric(df_perusahaan['Minimal_IPK'], errors='coerce')
df_perusahaan['Minimal_IPK'] = df_perusahaan['Minimal_IPK'].fillna(0)

print('Nilai unik Minimal_IPK sesudah :', df_perusahaan['Minimal_IPK'].unique())

Nilai unik Minimal_IPK sebelum: ['3.00' 'Tidak ada syarat khusus']
Nilai unik Minimal_IPK sesudah : [3. 0.]


### 5.4 Trim whitespace kolom string

In [24]:
str_cols_p = ['Nama_Perusahaan', 'Profil_Perusahaan', 'Bidang_Industri',
              'Posisi_Magang', 'Job_Description', 'Skill_Dibutuhkan',
              'Teknologi_Digunakan', 'Status_Paid']

for col in str_cols_p:
    df_perusahaan[col] = df_perusahaan[col].astype(str).str.strip()

print('✅ Whitespace dibersihkan')

✅ Whitespace dibersihkan


### 5.5 Normalisasi separator Skill & Teknologi perusahaan

In [25]:
df_perusahaan['Skill_Dibutuhkan']    = df_perusahaan['Skill_Dibutuhkan'].apply(normalize_separator)
df_perusahaan['Teknologi_Digunakan'] = df_perusahaan['Teknologi_Digunakan'].apply(normalize_separator)

print('✅ Separator dinormalisasi')
print('\nContoh Skill Dibutuhkan    :', df_perusahaan['Skill_Dibutuhkan'].iloc[0])
print('Contoh Teknologi Digunakan :', df_perusahaan['Teknologi_Digunakan'].iloc[0])

✅ Separator dinormalisasi

Contoh Skill Dibutuhkan    : Pemrograman dasar, logika sistem, komunikasi, kemampuan analisis
Contoh Teknologi Digunakan : Visual Basic / VB.NET, MySQL, atau teknologi web sesuai kebutuhan proyek


### 5.6 Normalisasi Status Paid

In [26]:
df_perusahaan['Status_Paid'] = df_perusahaan['Status_Paid'].str.capitalize()
df_perusahaan['Status_Paid'] = df_perusahaan['Status_Paid'].str.replace(' (tergantung departemen)', '', regex=False).str.strip()
df_perusahaan['Status_Paid'] = df_perusahaan['Status_Paid'].str.replace('Unpaid / paid', 'Flexible', regex=False)
print('Nilai unik Status_Paid:')
print(df_perusahaan['Status_Paid'].value_counts())

Nilai unik Status_Paid:
Status_Paid
Paid        25
Unpaid       4
Flexible     1
Name: count, dtype: int64


### 5.7 Cek duplikat Perusahaan

In [27]:
dup_p = df_perusahaan.duplicated(subset=['Nama_Perusahaan', 'Posisi_Magang'])
print(f'Baris duplikat (nama + posisi sama): {dup_p.sum()}')

if dup_p.sum() > 0:
    display(df_perusahaan[dup_p][['Nama_Perusahaan','Posisi_Magang']])
else:
    print('✅ Tidak ada duplikat')

Baris duplikat (nama + posisi sama): 0
✅ Tidak ada duplikat


### 5.9 Normalisasi Bidang Industri

In [28]:
def normalize_bidang_industri(bidang):
    bidang = bidang.lower()
    if 'swasta nasional' in bidang:
        return 'swasta nasional'
    elif 'swasta' in bidang:
        return 'swasta'
    elif 'instansi pendidikan' in bidang or 'perguruan tinggi' in bidang:
        return 'instansi pendidikan'
    elif 'bumn' in bidang or 'badan usaha milik negara' in bidang:
        return 'BUMN'
    else:
        return 'lain-lain'

df_perusahaan['Bidang_Industri'] = df_perusahaan['Bidang_Industri'].apply(normalize_bidang_industri)

print('Nilai unik Bidang_Industri setelah normalisasi:')
print(df_perusahaan['Bidang_Industri'].value_counts())

Nilai unik Bidang_Industri setelah normalisasi:
Bidang_Industri
swasta nasional        19
swasta                  4
instansi pendidikan     4
BUMN                    3
Name: count, dtype: int64


### 5.8 Preview hasil akhir PERUSAHAAN

In [29]:
print(f'Total baris akhir  : {df_perusahaan.shape[0]}')
print(f'Total null tersisa : {df_perusahaan.isnull().sum().sum()}')
df_perusahaan.head()

Total baris akhir  : 30
Total null tersisa : 0


,ID_Perusahaan,Nama_Perusahaan,Profil_Perusahaan,Bidang_Industri,Posisi_Magang,Minimal_IPK,Job_Description,Skill_Dibutuhkan,Teknologi_Digunakan,Durasi_Bulan,Status_Paid
0,P001,PT Indoprima Gemilang,PT Indoprima Gemilang berdiri sejak 1976 di Su...,swasta nasional,IT/System Development Intern,3.0,Membantu pengembangan dan pemeliharaan sistem ...,"Pemrograman dasar, logika sistem, komunikasi, ...","Visual Basic / VB.NET, MySQL, atau teknologi w...",6,Paid
1,P002,Pengembangan SIPP – PT Link Apisindo Media & D...,PT Link Apisindo Media adalah perusahaan IT di...,swasta nasional,Backend Engineer / Frontend Engineer / AI Engi...,3.0,"Backend: pengembangan API, integrasi data, pen...","Backend: REST API, database management, system...","Backend: Laravel/FastAPI, MySQL/PostgreSQL, Do...",5,Paid
2,P003,Pengembangan Platform Satu Peta Jatim – PT Lin...,PT Link Apisindo Media berkolaborasi dengan Di...,swasta nasional,Data Engineer / Backend Developer / Frontend D...,3.0,"Data: ETL data spasial/non-spasial, integrasi ...","Data: ETL, GIS, database. Backend: Python, RES...","PostGIS, Python (FastAPI), JavaScript, OpenLay...",5,Paid
3,P004,PT ARM Solusi,"PT ARM Solusi berdiri sejak 2001, merupakan pe...",swasta nasional,Blockchain Engineer / Backend Developer / Data...,3.0,"Blockchain: pengembangan Hyperledger Fabric, s...","Blockchain: distributed systems, smart contrac...","Hyperledger Fabric, Python, Node.js / React, S...",5,Paid
4,P005,Pengembangan Platform Satu Peta Jatim – Batch ...,Batch kedua dari kolaborasi PT Link Apisindo M...,swasta nasional,Data Engineer / Backend Developer / Frontend D...,3.0,"Data: ETL data spasial & non-spasial, harmonis...","ETL, GIS, Python, REST API, JavaScript, WebGIS...","PostGIS, Python (FastAPI), OpenLayers/Leaflet,...",6,Paid


---
## 6. Ringkasan Akhir Preprocessing

In [30]:
print('=' * 55)
print('RINGKASAN HASIL PREPROCESSING')
print('=' * 55)

print('\n📋 MAHASISWA')
print(f'  Total baris    : {df_mahasiswa.shape[0]}')
print(f'  Kolom          : {list(df_mahasiswa.columns)}')
print(f'  Null tersisa   : {df_mahasiswa.isnull().sum().sum()}')
print(f'  IPK range      : {df_mahasiswa["IPK"].min()} – {df_mahasiswa["IPK"].max()}')
print(f'  Distribusi Prodi:')
print(df_mahasiswa['Prodi'].value_counts().to_string())

print('\n🏢 PERUSAHAAN')
print(f'  Total baris    : {df_perusahaan.shape[0]}')
print(f'  Kolom          : {list(df_perusahaan.columns)}')
print(f'  Null tersisa   : {df_perusahaan.isnull().sum().sum()}')
print(f'  Status Paid:')
print(df_perusahaan['Status_Paid'].value_counts().to_string())

print('\n✅ Preprocessing selesai!')

RINGKASAN HASIL PREPROCESSING

📋 MAHASISWA
  Total baris    : 305
  Kolom          : ['NIM', 'Nama', 'Prodi', 'Teknologi', 'Minat_Bidang', 'Skill', 'IPK']
  Null tersisa   : 0
  IPK range      : 2.43 – 3.97
  Distribusi Prodi:
Prodi
D4 Teknik Informatika         192
D4 Sistem Informasi Bisnis    113

🏢 PERUSAHAAN
  Total baris    : 30
  Kolom          : ['ID_Perusahaan', 'Nama_Perusahaan', 'Profil_Perusahaan', 'Bidang_Industri', 'Posisi_Magang', 'Minimal_IPK', 'Job_Description', 'Skill_Dibutuhkan', 'Teknologi_Digunakan', 'Durasi_Bulan', 'Status_Paid']
  Null tersisa   : 0
  Status Paid:
Status_Paid
Paid        25
Unpaid       4
Flexible     1

✅ Preprocessing selesai!


---
## 7. Simpan & Download Hasil Preprocessing

In [31]:
output_file = 'PBL_DATA_preprocessed.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_mahasiswa.to_excel(writer, sheet_name='MAHASISWA', index=False)
    df_perusahaan.to_excel(writer, sheet_name='PERUSAHAAN', index=False)

print(f'✅ File tersimpan: {output_file}')

# Download otomatis ke komputer
files.download(output_file)

✅ File tersimpan: PBL_DATA_preprocessed.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>